# Task 3 — Cleaning Data

**Goal:** Take a real-world messy dataset (Titanic passenger data) and clean it step by step, documenting every decision I make.

Dataset: Titanic passenger dataset (891 rows) — a classic "messy" dataset with missing values, mixed formatting and some outliers.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("titanic.csv")
df.shape

(891, 12)

In [2]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Data Quality Report (before cleaning)

In [3]:
quality_report_before = pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2)
})
quality_report_before

,dtype,nulls,null_pct
PassengerId,int64,0,0.00
Survived,int64,0,0.00
Pclass,int64,0,0.00
Name,str,0,0.00
Sex,str,0,0.00
Age,float64,177,19.87
SibSp,int64,0,0.00
Parch,int64,0,0.00
Ticket,str,0,0.00
Fare,float64,0,0.00


In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Total rows:", len(df))

Duplicate rows: 0
Total rows: 891


**Observation:** `Cabin` is missing for most rows (~77%), `Age` is missing for about 20% of rows, and `Embarked` has just 2 missing values. No exact duplicate rows in this dataset. `Age`, `Fare` are numeric, `Sex`, `Embarked`, `Name`, `Ticket`, `Cabin` are text columns.

## 2. Missing data handling

I'm handling each column differently based on how much is missing and what makes sense for that column:

- **Age** (~20% missing, numeric): fill with the **median** age. Median is safer than mean here because age has some outliers on the older side, and median isn't pulled by them.
- **Embarked** (only 2 missing): fill with the **mode** (most common port) since it's just 2 rows, this won't skew anything.
- **Cabin** (~77% missing): too many missing to impute meaningfully. Instead of dropping the column entirely (it still carries a signal — passengers *with* a cabin number were more likely 1st class), I'll convert it into a simple `Has_Cabin` flag (1 if known, 0 if missing) and drop the original noisy text column.

In [5]:
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df["Has_Cabin"] = df["Cabin"].notnull().astype(int)
df = df.drop(columns=["Cabin"])

df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Has_Cabin      0
dtype: int64

## 3. Duplicate removal

In [6]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)
print(f"Removed {before_rows - after_rows} duplicate rows")

Removed 0 duplicate rows


**Observation:** 0 duplicate rows were found/removed — this dataset was already unique per PassengerId. Still ran the check since it's a mandatory step for any cleaning pipeline, even when the result is "nothing to remove".

## 4. Standardisation of inconsistent formatting

In [7]:
df["Sex"].unique()

<StringArray>
['male', 'female']
Length: 2, dtype: str

In [8]:
# Sex column is already consistent lowercase ('male'/'female') in this dataset,
# but I'm still standardising it explicitly in case of stray casing/whitespace —
# this is good practice even if it looks like a no-op here.
df["Sex"] = df["Sex"].str.strip().str.lower()

df["Embarked"].unique()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str

In [9]:
# Embarked is stored as single letters (S/C/Q) — mapping to full port names
# makes the data more readable for anyone using this cleaned file later.
embarked_map = {"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}
df["Embarked"] = df["Embarked"].map(embarked_map)
df["Embarked"].value_counts()

Embarked
Southampton    646
Cherbourg      168
Queenstown      77
Name: count, dtype: int64

**Observation:** Sex values were already clean and consistent. Embarked was recoded from single-letter codes to full port names for readability.

## 5. Outlier detection (IQR method)

In [10]:
def iqr_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)], lower, upper

fare_outliers, fare_lo, fare_hi = iqr_outliers(df["Fare"])
age_outliers, age_lo, age_hi = iqr_outliers(df["Age"])

print(f"Fare: {len(fare_outliers)} outliers outside [{fare_lo:.2f}, {fare_hi:.2f}]")
print(f"Age:  {len(age_outliers)} outliers outside [{age_lo:.2f}, {age_hi:.2f}]")

Fare: 116 outliers outside [-26.72, 65.63]
Age:  66 outliers outside [2.50, 54.50]


**Decision:** `Fare` has a good number of high-value outliers (expensive first-class tickets) — these are **real, valid values**, not data errors, so I'm choosing to **cap** them at the upper IQR bound rather than deleting those passengers (deleting would lose real 1st-class passenger records). For `Age`, the few outliers are elderly passengers, which are also plausible real ages, so I'm leaving `Age` untouched and only capping `Fare`.

In [11]:
df["Fare"] = np.where(df["Fare"] > fare_hi, fare_hi, df["Fare"])
df["Fare"].describe()

count    891.000000
mean      24.046813
std       20.481625
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max       65.634400
Name: Fare, dtype: float64

## 6. Data type correction

In [12]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
Has_Cabin        int64
dtype: object

In [13]:
df["Survived"] = df["Survived"].astype("category")
df["Pclass"] = df["Pclass"].astype("category")
df["PassengerId"] = df["PassengerId"].astype(str)

df.dtypes

PassengerId         str
Survived       category
Pclass         category
Name                str
Sex                 str
Age             float64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Embarked            str
Has_Cabin         int64
dtype: object

**Observation:** `Survived` and `Pclass` are really categories (0/1 and 1/2/3), not continuous numbers, so recoding them as `category` dtype makes downstream analysis clearer. `PassengerId` is an identifier, not a quantity to do math on, so it's converted to string.

## 7. Before vs After summary

In [14]:
quality_report_after = pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isnull().sum()
})

summary = pd.DataFrame({
    "Metric": ["Row count", "Duplicate rows", "Total nulls", "Columns"],
    "Before": [before_rows, "0 (checked)", quality_report_before["nulls"].sum(), quality_report_before.shape[0]],
    "After":  [len(df), df.duplicated().sum(), df.isnull().sum().sum(), df.shape[1]]
})
summary

,Metric,Before,After
0,Row count,891,891
1,Duplicate rows,0 (checked),0
2,Total nulls,866,0
3,Columns,12,12


**Observation:** Null count dropped from a few hundred combined (mostly Age + Cabin) down to 0. Row count stayed the same (no rows were deleted, only capped/imputed/recoded) since none of the missingness or outliers justified throwing away data.

## 8. Save cleaned dataset

In [15]:
df.to_csv("titanic_cleaned.csv", index=False)
print("Saved titanic_cleaned.csv —", df.shape)
df.head()

Saved titanic_cleaned.csv — (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Has_Cabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,Southampton,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,65.6344,Cherbourg,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,Southampton,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,Southampton,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,Southampton,0


## Conclusion

The main cleaning decisions were: median-impute Age, mode-impute Embarked, convert the very-sparse Cabin column into a binary flag instead of dropping it outright, cap extreme Fare outliers instead of deleting rows, and fix dtypes for categorical/ID columns. No rows were removed — every decision favoured keeping data while fixing quality issues.